# Урок 10 - AI агенти в продукция

В този урок ще научите **продукционни модели** за AI агенти, използвайки Microsoft Agent Framework (Python). Обсъждаме:

- **Наблюдаемост** — добавяне на времеви измервания и логване на взаимодействията на агента
- **Оценка** — използване на агент-оценител за оценка на качеството на отговорите
- **Управление на разходите** — стратегии за оптимизация на токените и избор на модел

Сценарият е **туристически агент**, който помага на потребителите да планират пътувания, с наблюдение и оценка, положени върху него.


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Производствени съображения

Преместването на AI агенти от прототипи към производство изисква внимателно внимание към три стълба:

1. **Наблюдение** — Трябва да имате видимост върху това какво прави агентът, колко време отнема и кои инструменти използва. Без проследяване и логване, отстраняването на проблеми в производството е почти невъзможно.

2. **Оценка** — Автоматизираните проверки за качество гарантират, че отговорите на агента остават точни, пълни и полезни с течение на времето. Оценителен агент може да присъди оценки на отговорите спрямо определени критерии.

3. **Управление на разходите** — Използването на токени има пряко влияние върху разходите. Стратегии като оптимизиране на подканите, избор на модел и кеширане помагат да се поддържат разходите под контрол без да се жертва качеството.


## Създаване на наблюдаван агент

Дефинираме инструменти за пътуване и обвиваме извикването на агента с измерване на време, за да можем да наблюдаваме забавянето. В продукция бихте интегрирали с OpenTelemetry или подобна система за проследяване.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Шаблони за оценка

Често срещан производствен шаблон е да се използва втори агент като **оценител**. Оценителят оценява отговора на първичния агент спрямо предварително зададени критерии като пълнота, точност и полезност.

Това позволява:
- Автоматизирани контролни точки за качество преди отговорите да достигнат до потребителите
- Откриване на регресии при промени в подканите или моделите
- Непрекъснат мониторинг на представянето на агента с течение на времето


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегии за управление на разходите

Контролът на разходите е критичен за производствените AI агенти. Ето ключови стратегии:

| Стратегия | Описание |
|---|---|
| **Оптимизация на подканите** | Дръжте системните инструкции кратки. Премахнете излишен контекст, за да намалите входните токени. |
    "| **Избор на модел** | Използвайте по-малки, по-евтини модели (например GPT-5-mini) за прости задачи като класификация или извличане, и запазете по-големите модели за сложни разсъждения. |\n",
| **Кеширане** | Кеширайте резултатите от инструментите и често задаваните заявки, за да избегнете излишни API повиквания. |
| **Бюджет за токени** | Задайте лимити за `max_tokens`, за да предотвратите неочаквано дълги отговори. |
| **Групиране** | Групирайте няколко потребителски заявки в едно API повикване, когато е възможно. |

На практика работи подход с нива: пренасочвайте праволинейните заявки към бърз и евтин модел и ескалирайте само сложните заявки към по-способен модел.


## Резюме

В този урок научихте как да:

1. **Добавите наблюдаваемост** към взаимодействията с агента чрез измерване на времето и логване, като по този начин се създават основи за проследяване и мониторинг.
2. **Оценявате отговорите на агента** автоматично чрез агент-оценител, който дава оценки за пълнота, точност и полезност.
3. **Управлявате разходите** чрез оптимизация на подкана, избор на модел, кеширане и бюджетиране на токени.

Тези производствени модели помагат да гарантирате, че вашите AI агенти са надеждни, измерими и ефективни по отношение на разходите в мащаб.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
